Copyright 2026 Snowflake Inc.
SPDX-License-Identifier: Apache-2.0

Licensed under the Apache License, Version 2.0 (the "License");
you may not use this file except in compliance with the License.
You may obtain a copy of the License at

http://www.apache.org/licenses/LICENSE-2.0

Unless required by applicable law or agreed to in writing, software
distributed under the License is distributed on an "AS IS" BASIS,
WITHOUT WARRANTIES OR CONDITIONS OF ANY KIND, either express or implied.
See the License for the specific language governing permissions and
limitations under the License.

# Exercise 3.1: Ingestion Patterns and Concurrency

Write patterns in Iceberg on the NYC Taxi dataset:
- **Batch vs streaming** ingestion
- **Copy-on-Write (COW) vs Merge-on-Read (MOR)** tradeoffs
- **Row-level operations** — UPDATE / DELETE / MERGE
- **Concurrency** — optimistic locking, conflicts, retries

⚠️ **Spark Connect**: one Spark server in the background, notebooks are thin clients. If `ConnectionRefusedError` appears, check `docker logs jupyter-spark` or restart with `docker compose restart jupyter`.

## Initialize Spark Session

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *
import time
from datetime import datetime, timedelta
import random

spark = SparkSession.builder \
    .appName("IngestionPatterns") \
    .getOrCreate()

print(f"Spark {spark.version} initialized!")

## Create Namespace

In [ ]:
spark.sql("CREATE NAMESPACE IF NOT EXISTS polaris.ingestion")
print("Namespace 'ingestion' created!")

## Download NYC Taxi Data

We'll use NYC Yellow Taxi trip data for **June through August 2023**.

In [ ]:
import boto3
from botocore.client import Config
import urllib.request
import os

s3_client = boto3.client(
    's3',
    endpoint_url='http://minio:9000',
    aws_access_key_id=os.environ.get('MINIO_ROOT_USER', 'admin'),
    aws_secret_access_key=os.environ.get('MINIO_ROOT_PASSWORD', 'password'),
    config=Config(signature_version='s3v4'),
    region_name='us-east-1'
)

base_url = "https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2023-{:02d}.parquet"
bucket = "warehouse"

for month in [6, 7, 8]:
    filename = f"yellow_tripdata_2023-{month:02d}.parquet"
    key = f"raw/{filename}"
    try:
        s3_client.head_object(Bucket=bucket, Key=key)
        print(f"{filename} already in MinIO, skipping download")
    except:
        local_path = f"/tmp/{filename}"
        print(f"Downloading {filename} (~45MB)...")
        urllib.request.urlretrieve(base_url.format(month), local_path)
        s3_client.upload_file(local_path, bucket, key)
        os.remove(local_path)
        print(f"  Uploaded to s3a://{bucket}/{key}")

print("\nAll taxi data ready in MinIO!")

## Part 1: Batch vs Streaming Ingestion Patterns

### Batch Ingestion Pattern

Traditional databases manage write frequency for you; Iceberg doesn't. Every commit creates a snapshot and potentially new files, so write frequency directly drives metadata accumulation and file layout. That makes the batch (large, infrequent) vs streaming (small, frequent) split matter.

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.ingestion.taxi_trips")

spark.sql("""
    CREATE TABLE polaris.ingestion.taxi_trips
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    AS SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet` WHERE 1=0
""")

print("Taxi trips table created!")

In [ ]:
june_df = spark.read.parquet("s3a://warehouse/raw/yellow_tripdata_2023-06.parquet")

print(f"Batch loading {june_df.count():,} June trips...")

start = time.time()
june_df.writeTo("polaris.ingestion.taxi_trips").append()
batch_time = time.time() - start

print(f"Batch write completed in {batch_time:.2f} seconds")

In [ ]:
print("Snapshots after batch write:")
spark.sql("""
    SELECT committed_at, operation, summary['added-records'] as records_added
    FROM polaris.ingestion.taxi_trips.snapshots
    ORDER BY committed_at
""").show(truncate=False)

### Streaming Helper

> Implementation detail — uses Spark Structured Streaming to read from a staging dir into an Iceberg table in micro-batches. Tweak the params to control behavior.

In [ ]:
import shutil, os

def run_streaming_ingest(source_path, table_name,
                         num_source_files=10, max_files_per_trigger=2):
    """Stream parquet data into an Iceberg table using Spark Structured Streaming.

    Parameters:
        source_path:           S3 path to a source parquet file
        table_name:            Target Iceberg table (must already exist)
        num_source_files:      Split source into this many small files for streaming
        max_files_per_trigger: Files read per micro-batch (fewer = more snapshots)

    Returns the number of new snapshots created.
    """
    staging = "s3a://warehouse/staging/streaming_input"
    checkpoint = "/tmp/iceberg_streaming_checkpoint"

    if os.path.exists(checkpoint):
        shutil.rmtree(checkpoint)

    df = spark.read.parquet(source_path)
    schema = df.schema
    total_rows = df.count()

    print(f"Staging {total_rows:,} rows as {num_source_files} files...")
    df.repartition(num_source_files).write.mode("overwrite").parquet(staging)

    before = spark.sql(
        f"SELECT COUNT(*) FROM {table_name}.snapshots"
    ).collect()[0][0]

    print(f"Streaming with maxFilesPerTrigger={max_files_per_trigger} "
          f"(expect ~{num_source_files // max_files_per_trigger} micro-batches)...")
    start = time.time()

    query = (spark.readStream
             .schema(schema)
             .option("maxFilesPerTrigger", max_files_per_trigger)
             .parquet(staging)
             .writeStream
             .format("iceberg")
             .outputMode("append")
             .option("checkpointLocation", checkpoint)
             .toTable(table_name))

    query.processAllAvailable()
    query.stop()

    elapsed = time.time() - start
    after = spark.sql(
        f"SELECT COUNT(*) FROM {table_name}.snapshots"
    ).collect()[0][0]
    new_snapshots = after - before

    print(f"Done in {elapsed:.2f}s, created {new_snapshots} new snapshots")
    return new_snapshots

print("Streaming helper loaded!")

### Streaming Ingestion Pattern

Small frequent writes. Typical for real-time streaming.

In [ ]:
streaming_snapshots = run_streaming_ingest(
    source_path="s3a://warehouse/raw/yellow_tripdata_2023-07.parquet",
    table_name="polaris.ingestion.taxi_trips",
    num_source_files=10,
    max_files_per_trigger=2
)

In [ ]:
snapshot_count = spark.sql("""
    SELECT COUNT(*) as count FROM polaris.ingestion.taxi_trips.snapshots
""").collect()[0]['count']

print(f"Total snapshots: {snapshot_count}")

Streaming yields many small snapshots vs batch's single large one. Each snapshot adds metadata overhead (manifests, small files). Unlike a traditional DB, Iceberg commits are immutable — each writes new files to object storage. E3.2 covers the maintenance that fills this gap (compaction, metadata cleanup). Tradeoff: more frequent commits = fresher data, more maintenance.

### Try It: Adjust Streaming Parameters

Use `run_streaming_ingest` with different `num_source_files` and `max_files_per_trigger`. More micro-batches → more snapshots + small files.

In [ ]:
# Try with many small micro-batches (more snapshots, more files):
# run_streaming_ingest(
#     source_path="s3a://warehouse/raw/yellow_tripdata_2023-08.parquet",
#     table_name="polaris.ingestion.taxi_trips",
#     num_source_files=20,
#     max_files_per_trigger=1
# )

# Or fewer, larger micro-batches (fewer snapshots):
# run_streaming_ingest(
#     source_path="s3a://warehouse/raw/yellow_tripdata_2023-08.parquet",
#     table_name="polaris.ingestion.taxi_trips",
#     num_source_files=20,
#     max_files_per_trigger=10
# )

# Compare the results
# spark.sql("SELECT COUNT(*) as snapshots FROM polaris.ingestion.taxi_trips.snapshots").show()
# spark.sql("SELECT COUNT(*) as files FROM polaris.ingestion.taxi_trips.files").show()

## Part 2: Copy-on-Write vs Merge-on-Read

Iceberg stores immutable files on object storage — no in-place row updates. Two strategies:

- **COW** — rewrites the entire data file on update. Slower writes, faster reads.
- **MOR** — writes small delete files alongside originals. Faster writes, slower reads (merge at read time).

> Timings illustrate *relative* tradeoffs. Absolute numbers vary by machine; production clusters show bigger gaps.

### Create COW and MOR Tables

In [ ]:
for mode_suffix, mode_value in [('cow', 'copy-on-write'), ('mor', 'merge-on-read')]:
    spark.sql(f"DROP TABLE IF EXISTS polaris.ingestion.taxi_{mode_suffix}")
    spark.sql(f"""
        CREATE TABLE polaris.ingestion.taxi_{mode_suffix}
        USING iceberg
        TBLPROPERTIES (
            'write.update.mode' = '{mode_value}',
            'write.delete.mode' = '{mode_value}',
            'write.merge.mode' = '{mode_value}'
        )
        AS SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
    """)

row_count = spark.sql("SELECT COUNT(*) FROM polaris.ingestion.taxi_cow").collect()[0][0]
size_mb = spark.sql("""
    SELECT ROUND(SUM(file_size_in_bytes) / 1024 / 1024, 1)
    FROM polaris.ingestion.taxi_cow.files
""").collect()[0][0]
print(f"COW and MOR tables created with {row_count:,} rows each (~{size_mb} MB)")

### Compare UPDATE Performance

In [ ]:
start = time.time()
spark.sql("""
    UPDATE polaris.ingestion.taxi_cow
    SET fare_amount = fare_amount * 1.10
    WHERE trip_distance > 10
""")
cow_time = time.time() - start
print(f"COW UPDATE took {cow_time:.3f} seconds")

In [ ]:
start = time.time()
spark.sql("""
    UPDATE polaris.ingestion.taxi_mor
    SET fare_amount = fare_amount * 1.10
    WHERE trip_distance > 10
""")
mor_time = time.time() - start
print(f"MOR UPDATE took {mor_time:.3f} seconds")

if mor_time < cow_time:
    print(f"\nMOR was {cow_time/mor_time:.1f}x faster for writes")
else:
    print(f"\nCOW was {mor_time/cow_time:.1f}x faster for writes (small table effect)")

### Examine File Structure

In [ ]:
print("COW data files:")
spark.sql("""
    SELECT SUBSTRING(file_path, LENGTH(file_path) - LOCATE('/', REVERSE(file_path)) + 2) as filename,
           file_size_in_bytes, record_count
    FROM polaris.ingestion.taxi_cow.files
""").show(truncate=False)

In [ ]:
print("MOR data and delete files:")
spark.sql("""
    SELECT SUBSTRING(file_path, LENGTH(file_path) - LOCATE('/', REVERSE(file_path)) + 2) as filename,
           file_size_in_bytes, record_count
    FROM polaris.ingestion.taxi_mor.files
""").show(truncate=False)

**COW** has a **single data file** — the update rewrote the whole file (modified + unchanged rows).

**MOR** kept the original and wrote a small **delete file** listing changed row positions. At read time Spark skips those positions and merges in the replacement rows.

### Compare READ Performance

We sum a column to force Spark to read every value from every file, while returning only one number to the driver (avoids Python serialization bottleneck).

In [ ]:
scan_query = "SELECT SUM(fare_amount) FROM {}"

start = time.time()
spark.sql(scan_query.format("polaris.ingestion.taxi_cow")).collect()
cow_read = time.time() - start

start = time.time()
spark.sql(scan_query.format("polaris.ingestion.taxi_mor")).collect()
mor_read = time.time() - start

print(f"COW READ: {cow_read:.3f}s")
print(f"MOR READ: {mor_read:.3f}s")
if cow_read < mor_read:
    print(f"\nCOW reads were {mor_read/cow_read:.1f}x faster (no delete-file merge overhead)")
else:
    print(f"\nResults similar at this scale")

### Try It: How Does Selectivity Affect COW vs MOR?

The update touched rows where `trip_distance > 10` (small fraction). Try other `update_pct` values:

| `update_pct` | Filter | Rows |
|---|---|---|
| `1` | Longest 1% | ~33K |
| `10` | Longer-than-average | ~330K |
| `50` | Half the table | ~1.6M |
| `90` | Nearly all | ~3M |

**Prediction:** MOR wins at low % (small delete files). As % grows, COW's single-file rewrite competes because MOR accumulates large delete files that slow reads.

In [ ]:
def benchmark_cow_vs_mor(update_pct):
    """Recreate COW and MOR tables, update a percentage of rows, and compare write/read performance."""
    for mode_suffix, mode_value in [('cow', 'copy-on-write'), ('mor', 'merge-on-read')]:
        spark.sql(f"DROP TABLE IF EXISTS polaris.ingestion.taxi_{mode_suffix}")
        spark.sql(f"""
            CREATE TABLE polaris.ingestion.taxi_{mode_suffix}
            USING iceberg
            TBLPROPERTIES (
                'write.update.mode' = '{mode_value}',
                'write.delete.mode' = '{mode_value}',
                'write.merge.mode' = '{mode_value}'
            )
            AS SELECT * FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
        """)

    total = spark.sql("SELECT COUNT(*) FROM polaris.ingestion.taxi_cow").collect()[0][0]

    threshold = spark.sql(f"""
        SELECT PERCENTILE_APPROX(trip_distance, {(100 - update_pct) / 100.0})
        FROM polaris.ingestion.taxi_cow
    """).collect()[0][0]

    affected = spark.sql(f"""
        SELECT COUNT(*) FROM polaris.ingestion.taxi_cow WHERE trip_distance >= {threshold}
    """).collect()[0][0]
    print(f"Updating rows with trip_distance >= {threshold:.2f}")
    print(f"Affects {affected:,} / {total:,} rows ({100*affected/total:.1f}%)")
    print("=" * 60)

    start = time.time()
    spark.sql(f"UPDATE polaris.ingestion.taxi_cow SET fare_amount = fare_amount * 1.10 WHERE trip_distance >= {threshold}")
    cow_write = time.time() - start

    start = time.time()
    spark.sql(f"UPDATE polaris.ingestion.taxi_mor SET fare_amount = fare_amount * 1.10 WHERE trip_distance >= {threshold}")
    mor_write = time.time() - start

    scan_query = "SELECT SUM(fare_amount) FROM {}"
    start = time.time()
    spark.sql(scan_query.format("polaris.ingestion.taxi_cow")).collect()
    cow_read = time.time() - start

    start = time.time()
    spark.sql(scan_query.format("polaris.ingestion.taxi_mor")).collect()
    mor_read = time.time() - start

    print(f"\n{'Metric':<15} {'COW':>10} {'MOR':>10} {'Winner':>10}")
    print("-" * 50)
    print(f"{'Write':.<15} {cow_write:>9.3f}s {mor_write:>9.3f}s {'MOR' if mor_write < cow_write else 'COW':>10}")
    print(f"{'Read':.<15} {cow_read:>9.3f}s {mor_read:>9.3f}s {'COW' if cow_read < mor_read else 'MOR':>10}")

print("benchmark_cow_vs_mor() helper ready")

In [ ]:
# benchmark_cow_vs_mor(update_pct=???)  # <-- Try: 1, 10, 50, 90

## Part 3: Row-Level Operations

These trigger **row-level** changes — Iceberg identifies and modifies individual rows within files (via COW or MOR):

| Op | Does | Example |
|---|---|---|
| `DELETE` | Remove matching rows | `DELETE FROM t WHERE fare < 0` |
| `UPDATE` | Modify matching rows | `UPDATE t SET fare = fare*1.1 WHERE ...` |
| `MERGE INTO` | Upsert | `MERGE INTO t USING src ON ... WHEN MATCHED THEN UPDATE WHEN NOT MATCHED THEN INSERT` |

Same mechanism underneath: scan affected files via column metrics, apply via COW/MOR, commit a new snapshot atomically.

### DELETE Operations

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.ingestion.taxi_partitioned")

spark.sql("""
    CREATE TABLE polaris.ingestion.taxi_partitioned
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    AS SELECT VendorID, tpep_pickup_datetime, passenger_count, trip_distance,
              fare_amount, tip_amount, total_amount
       FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
""")

count = spark.sql("SELECT COUNT(*) FROM polaris.ingestion.taxi_partitioned").collect()[0][0]
print(f"Partitioned table created with {count:,} trips")

In [ ]:
bad_fares = spark.sql("""
    SELECT COUNT(*) FROM polaris.ingestion.taxi_partitioned WHERE fare_amount <= 0
""").collect()[0][0]

start = time.time()
spark.sql("DELETE FROM polaris.ingestion.taxi_partitioned WHERE fare_amount <= 0")
delete_time = time.time() - start

remaining = spark.sql("SELECT COUNT(*) FROM polaris.ingestion.taxi_partitioned").collect()[0][0]
print(f"Deleted {bad_fares:,} bad-fare trips in {delete_time:.3f} seconds")
print(f"Remaining: {remaining:,} trips")

### Metadata-Only Deletes

When **every row in a file** matches the delete predicate, Iceberg drops the whole file from metadata — **no read, no rewrite**. Works because each file stores per-column min/max stats; if the predicate range fully covers a file's range, the file is droppable. Happens automatically.

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.ingestion.taxi_meta_delete")

spark.sql("""
    CREATE TABLE polaris.ingestion.taxi_meta_delete
    USING iceberg
    PARTITIONED BY (days(tpep_pickup_datetime))
    AS SELECT VendorID, tpep_pickup_datetime, passenger_count,
              trip_distance, fare_amount, tip_amount, total_amount
       FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
""")

count = spark.sql("SELECT COUNT(*) FROM polaris.ingestion.taxi_meta_delete").collect()[0][0]
partitions = spark.sql("""
    SELECT COUNT(DISTINCT DATE(tpep_pickup_datetime)) FROM polaris.ingestion.taxi_meta_delete
""").collect()[0][0]
print(f"Table created: {count:,} trips across {partitions} day-partitions (one file per day)")

One file per day-partition. Inspect the `readable_metrics` — specifically `tpep_pickup_datetime` min/max — which Iceberg uses to decide if a file can be dropped without reading it.

In [ ]:
print("File-level readable metrics for tpep_pickup_datetime:\n")

spark.sql("""
    SELECT
        SUBSTRING(file_path, LENGTH(file_path) - LOCATE('/', REVERSE(file_path)) + 2) AS file_name,
        record_count,
        readable_metrics.tpep_pickup_datetime.lower_bound AS pickup_min,
        readable_metrics.tpep_pickup_datetime.upper_bound AS pickup_max
    FROM polaris.ingestion.taxi_meta_delete.files
    ORDER BY pickup_min
    LIMIT 10
""").show(truncate=False)

Each file's `pickup_min`/`pickup_max` falls within a single day. `DELETE WHERE tpep_pickup_datetime >= '2023-06-01' AND < '2023-06-02'` → Iceberg sees the June 1 file's range is fully covered → drops it with **no scan, no rewrite**.

In [ ]:
rows_before = spark.sql("SELECT COUNT(*) FROM polaris.ingestion.taxi_meta_delete").collect()[0][0]
june1_rows = spark.sql("""
    SELECT COUNT(*) FROM polaris.ingestion.taxi_meta_delete
    WHERE tpep_pickup_datetime >= '2023-06-01' AND tpep_pickup_datetime < '2023-06-02'
""").collect()[0][0]

start = time.time()
spark.sql("""
    DELETE FROM polaris.ingestion.taxi_meta_delete
    WHERE tpep_pickup_datetime >= '2023-06-01' AND tpep_pickup_datetime < '2023-06-02'
""")
delete_time = time.time() - start

rows_after = spark.sql("SELECT COUNT(*) FROM polaris.ingestion.taxi_meta_delete").collect()[0][0]
print(f"Deleted {june1_rows:,} rows (entire June 1 partition) in {delete_time:.3f}s")
print(f"Rows: {rows_before:,} → {rows_after:,}")

In [ ]:
print("Snapshot summary for the delete operation:\n")

spark.sql("""
    SELECT operation,
           summary['deleted-data-files'] AS files_removed,
           summary['added-data-files'] AS files_added,
           summary['deleted-records'] AS records_deleted,
           summary['added-records'] AS records_added,
           summary['total-records'] AS total_remaining
    FROM polaris.ingestion.taxi_meta_delete.snapshots
    ORDER BY committed_at DESC
    LIMIT 1
""").show(truncate=False)

**files_removed = 1, files_added = 0** — no data read or rewritten, just a metadata reference removed. This is why partition-aligned deletes are so fast. A traditional DB would scan and mark individual rows.

### MERGE Operations (Upserts)

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.ingestion.taxi_corrections")

spark.sql("""
    CREATE TABLE polaris.ingestion.taxi_corrections
    USING iceberg
    AS SELECT VendorID, tpep_pickup_datetime, passenger_count,
              trip_distance, fare_amount, tip_amount, total_amount
       FROM parquet.`s3a://warehouse/raw/yellow_tripdata_2023-06.parquet`
       LIMIT 1000
""")

print(f"Corrections table created with 1,000 trips")

In [ ]:
corrections = spark.sql("""
    SELECT VendorID, tpep_pickup_datetime, passenger_count,
           trip_distance, fare_amount * 1.05 as fare_amount,
           tip_amount, total_amount * 1.05 as total_amount
    FROM polaris.ingestion.taxi_corrections
    LIMIT 100
""")
corrections.createOrReplaceTempView("fare_corrections")

print("Prepared 100 fare corrections (5% increase)")

In [ ]:
spark.sql("""
    MERGE INTO polaris.ingestion.taxi_corrections t
    USING fare_corrections s
    ON t.tpep_pickup_datetime = s.tpep_pickup_datetime
       AND t.VendorID = s.VendorID
       AND t.trip_distance = s.trip_distance
    WHEN MATCHED THEN UPDATE SET
        t.fare_amount = s.fare_amount,
        t.total_amount = s.total_amount
    WHEN NOT MATCHED THEN INSERT *
""")

print("MERGE completed!")

In [ ]:
print("Corrections table after MERGE:")
spark.sql("""
    SELECT COUNT(*) as total_rows,
           ROUND(AVG(fare_amount), 2) as avg_fare,
           ROUND(AVG(total_amount), 2) as avg_total
    FROM polaris.ingestion.taxi_corrections
""").show()

### Try It: Write Your Own MERGE

Build a small corrections dataset (e.g. adjust `tip_amount` for specific trips) and MERGE into `taxi_corrections` using both `WHEN MATCHED THEN UPDATE` and `WHEN NOT MATCHED THEN INSERT`. Check snapshot history.

In [ ]:
# Step 1: Create a corrections view. Change the column and factor
# IMPORTANT: The ON clause columns must uniquely identify rows. The combination
# (tpep_pickup_datetime, VendorID, trip_distance) may have duplicates in the
# full dataset, causing a MERGE_CARDINALITY_VIOLATION. With LIMIT 50 from
# an already-small table this is unlikely, but if you hit it, add more columns
# to the ON clause (e.g., fare_amount, tip_amount) to make matches unique.
# spark.sql("""
#     SELECT VendorID, tpep_pickup_datetime, passenger_count,
#            trip_distance, fare_amount, tip_amount * ??? as tip_amount, total_amount
#     FROM polaris.ingestion.taxi_corrections
#     LIMIT 50
# """).createOrReplaceTempView("my_corrections")

# Step 2: MERGE your corrections. Fill in the WHEN clauses
# spark.sql("""
#     MERGE INTO polaris.ingestion.taxi_corrections t
#     USING my_corrections s
#     ON t.tpep_pickup_datetime = s.tpep_pickup_datetime
#        AND t.VendorID = s.VendorID
#        AND t.trip_distance = s.trip_distance
#     WHEN MATCHED THEN UPDATE SET t.tip_amount = s.tip_amount
#     WHEN NOT MATCHED THEN INSERT *
# """)

# Step 3: Check snapshot history for your MERGE operation
# spark.sql("SELECT committed_at, operation FROM polaris.ingestion.taxi_corrections.snapshots ORDER BY committed_at").show(truncate=False)

## Part 4: Concurrency and Conflicts

Traditional DBs use **pessimistic** locks. Iceberg uses **optimistic concurrency**: writers proceed independently, conflicts checked only at commit.

Detection is **file-level**, using the same column min/max stats from earlier. On commit, Iceberg checks whether the new commit's affected files overlap files changed by concurrent commits — comparing the `WHERE`/`ON` predicate against file metrics. Two writes that can only touch different files won't conflict.

### Concurrency Helper

`ThreadPoolExecutor` runs Spark SQL concurrently — `spark.sql()` releases the GIL while the Spark Connect server processes the query. Plumbing detail, not Iceberg-specific.

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

def run_concurrent_sqls(statements):
    """Run N named SQL statements concurrently and report results.
    
    Args:
        statements: list of (name, sql_string) tuples
    Returns:
        list of (name, status, error_snippet) tuples
    """
    def _extract_message(e):
        """Pull the meaningful exception line from a PySpark exception."""
        msg = str(e)
        for line in msg.split("\n"):
            line = line.strip()
            for marker in ["CommitFailedException", "ValidationException", "CommitStateUnknownException"]:
                if marker in line:
                    idx = line.find(marker)
                    return line[idx:][:200]
        return msg.split("\n")[0][:200]

    def _exec(name, sql):
        try:
            spark.sql(sql)
            return (name, "OK", None)
        except Exception as e:
            msg = str(e)
            snippet = _extract_message(e)
            conflict_markers = ["CommitFailedException", "ValidationException", "conflicting files"]
            if any(m in msg for m in conflict_markers):
                return (name, "CONFLICT", snippet)
            return (name, "ERROR", snippet)

    results = []
    with ThreadPoolExecutor(max_workers=len(statements)) as pool:
        futures = {
            pool.submit(_exec, name, sql): name
            for name, sql in statements
        }
        for future in as_completed(futures):
            results.append(future.result())

    for name, status, err in sorted(results, key=lambda r: r[0]):
        print(f"  {name}: {status}")
        if err:
            print(f"    --> {err}")
    return results

print("run_concurrent_sqls() helper ready")

### Non-Conflicting Writes

Updates whose predicates resolve to **different files** succeed without conflict. Partitioning (one file per partition) is the common way; any predicate that excludes files via column metrics works.

In [ ]:
print("Concurrently updating trips on two different days...\n")

start = time.time()
results = run_concurrent_sqls([
    ("Writer A (June 1)", """
        UPDATE polaris.ingestion.taxi_partitioned
        SET fare_amount = fare_amount * 1.02
        WHERE tpep_pickup_datetime >= '2023-06-01' AND tpep_pickup_datetime < '2023-06-02'
    """),
    ("Writer B (June 15)", """
        UPDATE polaris.ingestion.taxi_partitioned
        SET fare_amount = fare_amount * 1.03
        WHERE tpep_pickup_datetime >= '2023-06-15' AND tpep_pickup_datetime < '2023-06-16'
    """),
])
elapsed = time.time() - start
print(f"\nTotal wall-clock time: {elapsed:.1f}s (both ran in parallel)")

Both succeeded — predicates resolved to different files (each day = its own partition file). Iceberg confirmed no file overlap between the two writers.

### Conflicting Writes

Two writers target the **same files**: both read the current snapshot, do their work, try to commit. First wins; second detects its read files were replaced → `CommitFailedException`.

In [ ]:
print("Concurrently updating the SAME partition (June 1) from two writers...\n")

results = run_concurrent_sqls([
    ("Writer A (+2%)", """
        UPDATE polaris.ingestion.taxi_partitioned
        SET fare_amount = fare_amount * 1.02
        WHERE tpep_pickup_datetime >= '2023-06-01' AND tpep_pickup_datetime < '2023-06-02'
    """),
    ("Writer B (+3%)", """
        UPDATE polaris.ingestion.taxi_partitioned
        SET fare_amount = fare_amount * 1.03
        WHERE tpep_pickup_datetime >= '2023-06-01' AND tpep_pickup_datetime < '2023-06-02'
    """),
])

One succeeded, one got `CommitFailedException` — exactly what optimistic concurrency guarantees. **No data corruption**; the loser just re-reads and retries.

**Best practice:** put predicates in `WHERE` / `MERGE ON` that let Iceberg narrow the conflict window via file-level metrics. Partition columns work best (one file per partition); any column with good min/max separation helps.

### Commit Retries and Concurrency

On failure Iceberg can **auto-retry**: refresh the base snapshot, retry. Count = `commit.retry.num-retries` (default **4**).

**What's retryable:**
- **Appends (INSERT)** — always safe to retry; output files are valid against any base snapshot.
- **COW update/delete** — failure is a data-level `ValidationException`, **not** retryable; the rewrite was against stale files and must redo from scratch.

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.ingestion.taxi_retry_demo")

spark.sql("""
    CREATE TABLE polaris.ingestion.taxi_retry_demo (
        writer_id INT, batch_id INT, value DOUBLE
    )
    USING iceberg
    TBLPROPERTIES ('commit.retry.num-retries' = '1')
""")

print("Table created with commit.retry.num-retries = 1")
print("Each writer gets 1 original attempt + 1 retry = 2 chances total.")
print("Launching 10 concurrent INSERTs...\n")

results = run_concurrent_sqls([
    (f"Writer {chr(65+i)}", f"""
        INSERT INTO polaris.ingestion.taxi_retry_demo VALUES ({i}, 1, {i * 1.1})
    """)
    for i in range(10)
])

ok_count = sum(1 for _, status, _ in results if status == "OK")
conflict_count = sum(1 for _, status, _ in results if status == "CONFLICT")
error_count = sum(1 for _, status, _ in results if status == "ERROR")
print(f"\nResults: {ok_count} succeeded, {conflict_count + error_count} failed")
if conflict_count + error_count > 0:
    print("Some writers exhausted their single retry and couldn't commit.")

With only 1 retry, some INSERTs lost the catalog race (the branch pointer moved). These are **appends** — no data-level conflict, just serialization. More retries would let all writers land. Bump `commit.retry.num-retries` to 20 and retry.

In [ ]:
spark.sql("""
    ALTER TABLE polaris.ingestion.taxi_retry_demo
    SET TBLPROPERTIES ('commit.retry.num-retries' = '20')
""")
print("Set commit.retry.num-retries = 20")
print("Re-running 10 concurrent INSERTs...\n")

results = run_concurrent_sqls([
    (f"Writer {chr(65+i)}", f"""
        INSERT INTO polaris.ingestion.taxi_retry_demo VALUES ({i}, 2, {i * 2.2})
    """)
    for i in range(10)
])

ok_count = sum(1 for _, status, _ in results if status == "OK")
conflict_count = sum(1 for _, status, _ in results if status == "CONFLICT")
error_count = sum(1 for _, status, _ in results if status == "ERROR")
print(f"\nResults: {ok_count} succeeded, {conflict_count + error_count} failed")
if conflict_count + error_count == 0:
    print("All writers succeeded with enough retries to serialize all 10 commits.")

total_rows = spark.sql("SELECT COUNT(*) FROM polaris.ingestion.taxi_retry_demo").collect()[0][0]
print(f"Table now has {total_rows} rows total")

With enough retries, all 10 concurrent INSERTs succeeded — each loser refreshed its base and re-committed.

**Key insight:** `commit.retry.num-retries` handles **catalog-level races** (branch-pointer contention) for appends/manifest updates. COW update/delete failures are `ValidationException` (data conflict) and require full re-execution — the rewrite was against stale files.

Related: `commit.retry.min-wait-ms` (100), `commit.retry.max-wait-ms` (60000), `commit.retry.total-timeout-ms` (1800000).

### MERGE and the ON Clause: Scoping Conflict Detection

`UPDATE` uses `WHERE` to pick affected files; `MERGE` uses the **`ON` clause**. If `ON` has predicates that exclude files via column metrics (partition col, or any column with useful min/max bounds), Iceberg scopes the merge — concurrent merges on non-overlapping file sets both succeed.

But if the file-scoping filter is buried in `WHEN MATCHED`, Iceberg **can't** use it — file selection happens *before* `WHEN` evaluation. Without a scoping filter in `ON`, it must assume the merge could touch **any file** → concurrent merges conflict even when they logically target different data.

In [ ]:
spark.sql("DROP TABLE IF EXISTS polaris.ingestion.taxi_merge_demo")

spark.sql("""
    CREATE TABLE polaris.ingestion.taxi_merge_demo (
        id INT, day STRING, amount DOUBLE
    )
    USING iceberg
    PARTITIONED BY (day)
""")

spark.sql("""
    INSERT INTO polaris.ingestion.taxi_merge_demo VALUES
    (1, '2023-06-01', 10.0), (2, '2023-06-01', 20.0),
    (3, '2023-06-02', 30.0), (4, '2023-06-02', 40.0)
""")

spark.sql("""
    SELECT id, day, amount * 1.05 AS new_amount
    FROM polaris.ingestion.taxi_merge_demo
""").createOrReplaceTempView("merge_source")

print("Merge demo table and source view ready")
spark.sql("SELECT * FROM polaris.ingestion.taxi_merge_demo ORDER BY id").show()

**Case 1: file-scoping predicate in ON.** The date range lets Iceberg scope each merge to its own files via metrics → two merges on different days should both succeed.

In [ ]:
print("Case 1: File-scoping predicate IN the ON clause\n")

results = run_concurrent_sqls([
    ("Merge A (June 1)", """
        MERGE INTO polaris.ingestion.taxi_merge_demo t
        USING merge_source s
        ON t.id = s.id AND t.day = '2023-06-01'
        WHEN MATCHED THEN UPDATE SET t.amount = s.new_amount
    """),
    ("Merge B (June 2)", """
        MERGE INTO polaris.ingestion.taxi_merge_demo t
        USING merge_source s
        ON t.id = s.id AND t.day = '2023-06-02'
        WHEN MATCHED THEN UPDATE SET t.amount = s.new_amount
    """),
])

Both succeeded — Iceberg used the ON-clause date predicate to confirm non-overlapping files.

**Case 2: predicate only in WHEN MATCHED.** ON has no file-scoping filter → Iceberg must treat the merge as potentially touching every file. Even though the two merges target different days, it can't prove non-overlap.

In [ ]:
print("Case 2: No file-scoping predicate in the ON clause (only in WHEN MATCHED)\n")

results = run_concurrent_sqls([
    ("Merge A (June 1)", """
        MERGE INTO polaris.ingestion.taxi_merge_demo t
        USING merge_source s
        ON t.id = s.id
        WHEN MATCHED AND t.day = '2023-06-01'
        THEN UPDATE SET t.amount = s.new_amount
    """),
    ("Merge B (June 2)", """
        MERGE INTO polaris.ingestion.taxi_merge_demo t
        USING merge_source s
        ON t.id = s.id
        WHEN MATCHED AND t.day = '2023-06-02'
        THEN UPDATE SET t.amount = s.new_amount
    """),
])

One merge conflicted, despite targeting different days. Without a file-scoping predicate in `ON`, conflict detection treated both as overlapping across every file.

**Takeaway:** put file-scoping predicates in the `ON` clause, not `WHEN`. Partition columns are most reliable; any column with good min/max separation helps. Same file-level metrics mechanism as metadata-only deletes.

### Try It: Find Non-Conflicting Ranges Without Partitioning

Partitioning isn't the only way — any predicate mapping exclusively to a subset of files works. Here's an **unpartitioned** table sorted by `fare_amount`, so each file has a distinct, non-overlapping fare range.

**Challenge:**
1. Inspect `readable_metrics` for each file's `fare_amount` min/max
2. Pick two ranges each inside a **different** single file
3. Run concurrent updates → both should succeed
4. Then two ranges overlapping the **same** file → one should conflict

In [ ]:
from pyspark.sql.functions import col

spark.sql("DROP TABLE IF EXISTS polaris.ingestion.taxi_sorted")

df = spark.read.parquet("s3a://warehouse/raw/yellow_tripdata_2023-06.parquet") \
    .select("VendorID", "tpep_pickup_datetime", "passenger_count",
            "trip_distance", "fare_amount", "tip_amount", "total_amount") \
    .repartitionByRange(5, col("fare_amount")) \
    .sortWithinPartitions("fare_amount")

df.writeTo("polaris.ingestion.taxi_sorted") \
    .using("iceberg") \
    .create()

print("Unpartitioned table created, sorted by fare_amount into 5 files")
print("Each file has a distinct, non-overlapping fare_amount range.\n")

spark.sql("""
    SELECT
        SUBSTRING(file_path, LENGTH(file_path) - LOCATE('/', REVERSE(file_path)) + 2) AS file,
        record_count,
        readable_metrics.fare_amount.lower_bound AS fare_min,
        readable_metrics.fare_amount.upper_bound AS fare_max
    FROM polaris.ingestion.taxi_sorted.files
    ORDER BY fare_min
""").show(truncate=False)

In [ ]:
# Step 1: Pick two fare ranges that each fall within a DIFFERENT file
#         (use the min/max values from the metrics above)
# range_a = "fare_amount >= ??? AND fare_amount < ???"
# range_b = "fare_amount >= ??? AND fare_amount < ???"

# Step 2: Concurrent updates to different files (should both succeed)
# run_concurrent_sqls([
#     ("Range A", f"UPDATE polaris.ingestion.taxi_sorted SET tip_amount = tip_amount + 0.01 WHERE {range_a}"),
#     ("Range B", f"UPDATE polaris.ingestion.taxi_sorted SET tip_amount = tip_amount + 0.01 WHERE {range_b}"),
# ])

# Step 3: Now try two ranges that overlap the SAME file (one should conflict)
# overlapping = "fare_amount >= ??? AND fare_amount < ???"
# run_concurrent_sqls([
#     ("Writer A", f"UPDATE polaris.ingestion.taxi_sorted SET tip_amount = tip_amount + 0.01 WHERE {overlapping}"),
#     ("Writer B", f"UPDATE polaris.ingestion.taxi_sorted SET tip_amount = tip_amount + 0.02 WHERE {overlapping}"),
# ])

## Cleanup

In [ ]:
# Optional: Drop tables
# for table in ['taxi_trips', 'taxi_cow', 'taxi_mor', 'taxi_partitioned',
#               'taxi_corrections', 'taxi_meta_delete', 'taxi_merge_demo',
#               'taxi_retry_demo', 'taxi_sorted']:
#     spark.sql(f"DROP TABLE IF EXISTS polaris.ingestion.{table}")
# print("Tables dropped!")